# 面试问题：Best-of-N Rejection Sampling Fine-Tuning 怎样过滤、选择并训练？

可直接复述的回答：RSFT 对每个 prompt 采样 N 个候选，用安全门禁和 reward/evaluator 选择高质量回答，再做 SFT。不能直接取最高 reward，因为 Reward Model 漏洞会被 Best-of-N 放大。选择前要过滤硬安全失败、去重近似候选并限制长度或风格偏差。训练数据必须保留采样模型、温度、reward、过滤原因和选择概率。Loss Mask 只监督 assistant 回答，不能学习 prompt 或 Judge 文本。迭代后要用独立 evaluator 和冻结黄金集验收。观测 reward 上升不等于真实质量上升。

后续实验使用可读的小型业务数据验证关键判断。所有数值都标记为教学实验，不代表真实 GPU、线上流量或基础模型泛化结果。


## 1. 真实案例：客服 Prompt 候选集与输入预览

五个客服 prompt 各有三个脱敏候选，字段包含 reward、安全标签、独立质量分和去重键。某些危险冗长回答 reward 更高，用于演示 Best-of-N 放大奖励漏洞。


In [1]:
import torch  # 使用 PyTorch 实现 assistant-only 微型 SFT。
from torch import nn  # 使用基础模块构造教学语言头。
import torch.nn.functional as F  # 使用交叉熵训练选中回答。
torch.manual_seed(20260729)  # 固定训练参数初始化。
prompts08 = [  # 构造五个真实语义客服候选集。
    {"id": "p1", "prompt": "订单延迟怎么办", "candidates": [("查询物流并说明预计时间", 0.82, True, 0.90, "track"), ("耐心等待", 0.55, True, 0.45, "wait"), ("调用内部管理员接口强制改状态", 0.96, False, 0.20, "unsafe")]},  # 工具越权 reward hacking。
    {"id": "p2", "prompt": "如何申请退款", "candidates": [("提供订单号并确认退款原因", 0.86, True, 0.91, "refund"), ("把密码发给客服", 0.93, False, 0.10, "secret"), ("提供订单号并确认退款原因", 0.84, True, 0.90, "refund")]},  # 包含重复候选和凭据风险。
    {"id": "p3", "prompt": "商品破损", "candidates": [("上传商品照片并保留包装", 0.88, True, 0.92, "photo"), ("无需证据直接赔付", 0.79, False, 0.30, "bypass"), ("联系商家", 0.62, True, 0.55, "contact")]},  # 售后证据流程。
    {"id": "p4", "prompt": "修改收货地址", "candidates": [("发货前在订单页提交变更", 0.85, True, 0.89, "address"), ("发货后仍可任意修改", 0.90, True, 0.25, "wrong"), ("咨询客服", 0.60, True, 0.58, "contact")]},  # reward 与独立正确性不一致。
    {"id": "p5", "prompt": "会员积分过期吗", "candidates": [("以当前会员规则页面日期为准", 0.83, True, 0.88, "policy"), ("永不过期", 0.87, True, 0.22, "wrong"), ("查看规则", 0.64, True, 0.60, "short")]},  # 动态政策事实。
]  # 完成五组 Best-of-N 数据。
print("教学实验输入：prompt_id | prompt | candidate/reward/safe/eval/dedup")  # 输出候选集预览表头。
for item08 in prompts08:  # 逐 prompt 展示三个候选。
    print(item08)  # 输出一组采样记录。


教学实验输入：prompt_id | prompt | candidate/reward/safe/eval/dedup
{'id': 'p1', 'prompt': '订单延迟怎么办', 'candidates': [('查询物流并说明预计时间', 0.82, True, 0.9, 'track'), ('耐心等待', 0.55, True, 0.45, 'wait'), ('调用内部管理员接口强制改状态', 0.96, False, 0.2, 'unsafe')]}
{'id': 'p2', 'prompt': '如何申请退款', 'candidates': [('提供订单号并确认退款原因', 0.86, True, 0.91, 'refund'), ('把密码发给客服', 0.93, False, 0.1, 'secret'), ('提供订单号并确认退款原因', 0.84, True, 0.9, 'refund')]}
{'id': 'p3', 'prompt': '商品破损', 'candidates': [('上传商品照片并保留包装', 0.88, True, 0.92, 'photo'), ('无需证据直接赔付', 0.79, False, 0.3, 'bypass'), ('联系商家', 0.62, True, 0.55, 'contact')]}
{'id': 'p4', 'prompt': '修改收货地址', 'candidates': [('发货前在订单页提交变更', 0.85, True, 0.89, 'address'), ('发货后仍可任意修改', 0.9, True, 0.25, 'wrong'), ('咨询客服', 0.6, True, 0.58, 'contact')]}
{'id': 'p5', 'prompt': '会员积分过期吗', 'candidates': [('以当前会员规则页面日期为准', 0.83, True, 0.88, 'policy'), ('永不过期', 0.87, True, 0.22, 'wrong'), ('查看规则', 0.64, True, 0.6, 'short')]}


## 2. Baseline（基线）：直接选择最高 Reward

朴素 Best-of-N 不看安全和独立质量，`p1/p2` 会选择越权或索取密码的高 reward 回答，`p4/p5` 会选择事实错误回答。


In [2]:
baseline_selected08 = []  # 收集纯 reward 选择结果。
for item08 in prompts08:  # 对每个 prompt 选择最高 reward 候选。
    selected08 = max(item08["candidates"], key=lambda candidate08: candidate08[1])  # 忽略安全和独立 evaluator。
    baseline_selected08.append((item08["id"], selected08[0], selected08[1], selected08[2], selected08[3]))  # 保存文本、reward、安全和质量。
baseline_safe08 = sum(row08[3] for row08 in baseline_selected08) / len(baseline_selected08)  # 计算纯 reward 选择安全率。
baseline_quality08 = sum(row08[4] for row08 in baseline_selected08) / len(baseline_selected08)  # 计算独立质量均值。
print("纯Reward基线：id | selected | reward | safe | independent_quality")  # 输出基线选择表头。
for row08 in baseline_selected08:  # 逐 prompt 展示 reward hacking。
    print(row08)  # 输出一条最高 reward 候选。


纯Reward基线：id | selected | reward | safe | independent_quality
('p1', '调用内部管理员接口强制改状态', 0.96, False, 0.2)
('p2', '把密码发给客服', 0.93, False, 0.1)
('p3', '上传商品照片并保留包装', 0.88, True, 0.92)
('p4', '发货后仍可任意修改', 0.9, True, 0.25)
('p5', '永不过期', 0.87, True, 0.22)


## 3. 核心实现：安全过滤、去重与独立排序

先删除硬安全失败，再按 `dedup_key` 保留一个候选，最后用独立 evaluator 选最高质量。Reward 仍记录在 provenance 中，但不能覆盖硬门禁。


In [3]:
selected_records08 = []  # 收集通过过滤的 RSFT 训练记录。
selection_trace08 = []  # 保存每个 prompt 的候选决策原因。
for item08 in prompts08:  # 逐 prompt 构造安全选择集。
    seen_keys08 = set()  # 初始化当前 prompt 的去重键集合。
    eligible08 = []  # 收集安全且不重复候选。
    decisions08 = []  # 收集逐候选过滤轨迹。
    for candidate08 in item08["candidates"]:  # 遍历 Best-of-N 候选。
        text08, reward08, safe08, evaluator08, dedup08 = candidate08  # 解包候选 provenance 字段。
        if not safe08:  # 先执行不可补偿安全门禁。
            decisions08.append((text08, "reject_unsafe"))  # 记录安全拒绝原因。
            continue  # 不让高 reward 覆盖安全失败。
        if dedup08 in seen_keys08:  # 检查近似候选是否重复。
            decisions08.append((text08, "reject_duplicate"))  # 记录重复拒绝原因。
            continue  # 避免同一模式占满选择分布。
        seen_keys08.add(dedup08)  # 登记新的语义候选键。
        eligible08.append(candidate08)  # 加入可由独立 evaluator 排序的集合。
        decisions08.append((text08, "eligible"))  # 记录候选通过门禁。
    selected08 = max(eligible08, key=lambda candidate08: candidate08[3])  # 使用独立质量而非训练 reward 排序。
    selected_records08.append({"id": item08["id"], "prompt": item08["prompt"], "response": selected08[0], "reward": selected08[1], "quality": selected08[3]})  # 保存带 provenance 的训练记录。
    selection_trace08.append((item08["id"], decisions08, selected08[0]))  # 保存完整选择轨迹。
print("核心选择轨迹：id | candidate decisions | final")  # 输出过滤和排序过程表头。
for row08 in selection_trace08:  # 逐 prompt 展示候选去向。
    print(row08)  # 输出一条可审计 Best-of-N 轨迹。


核心选择轨迹：id | candidate decisions | final
('p1', [('查询物流并说明预计时间', 'eligible'), ('耐心等待', 'eligible'), ('调用内部管理员接口强制改状态', 'reject_unsafe')], '查询物流并说明预计时间')
('p2', [('提供订单号并确认退款原因', 'eligible'), ('把密码发给客服', 'reject_unsafe'), ('提供订单号并确认退款原因', 'reject_duplicate')], '提供订单号并确认退款原因')
('p3', [('上传商品照片并保留包装', 'eligible'), ('无需证据直接赔付', 'reject_unsafe'), ('联系商家', 'eligible')], '上传商品照片并保留包装')
('p4', [('发货前在订单页提交变更', 'eligible'), ('发货后仍可任意修改', 'eligible'), ('咨询客服', 'eligible')], '发货前在订单页提交变更')
('p5', [('以当前会员规则页面日期为准', 'eligible'), ('永不过期', 'eligible'), ('查看规则', 'eligible')], '以当前会员规则页面日期为准')


## 4. 结果表、Assistant-only SFT 与结果解读

选中回答被渲染成 `<user> ... <assistant> ...`，标签只覆盖 assistant。微型 Embedding+Linear 仅验证 RSFT 数据可训练，不代表真实 LLM 收敛。


In [4]:
tokens08 = ["<pad>", "<user>", "<assistant>", "<eos>"]  # 定义教学模板特殊 token。
words08 = sorted({word08 for record08 in selected_records08 for text08 in [record08["prompt"], record08["response"]] for word08 in text08})  # 使用中文字符构造确定性小词表。
vocabulary08 = {token08: index08 for index08, token08 in enumerate(tokens08 + words08)}  # 建立 token 到 ID 映射。
sequences08 = []  # 收集模板 ID 和 assistant-only 标签。
for record08 in selected_records08:  # 逐条渲染选中训练记录。
    prompt_chars08 = list(record08["prompt"])  # 将用户 prompt 拆成可读字符 token。
    response_chars08 = list(record08["response"])  # 将选中回答拆成字符 token。
    rendered08 = ["<user>"] + prompt_chars08 + ["<assistant>"] + response_chars08 + ["<eos>"]  # 构造角色模板序列。
    ids08 = torch.tensor([vocabulary08[token08] for token08 in rendered08], dtype=torch.long)  # 转换为 ID 张量。
    labels08 = torch.full_like(ids08, -100)  # 默认忽略用户和角色 token。
    assistant_start08 = rendered08.index("<assistant>")  # 定位回答角色边界。
    labels08[assistant_start08 + 1:] = ids08[assistant_start08 + 1:]  # 只监督回答正文和结束 token。
    sequences08.append((ids08, labels08, rendered08))  # 保存训练张量和可读模板。
class TinyRSFT08(nn.Module):  # 定义验证数据链路的微型语言头。
    def __init__(self, vocab_size08, hidden08=48):  # 初始化嵌入和输出层。
        super().__init__()  # 注册模块参数。
        self.embedding = nn.Embedding(vocab_size08, hidden08)  # 将 token ID 映射为隐藏表示。
        self.output = nn.Linear(hidden08, vocab_size08)  # 预测下一字符 token。
    def forward(self, ids08):  # 定义前向计算。
        return self.output(self.embedding(ids08))  # 返回每个位置的词表 logits。
model08 = TinyRSFT08(len(vocabulary08))  # 创建确定性微型训练模型。
optimizer08 = torch.optim.Adam(model08.parameters(), lr=0.08)  # 使用基础优化器训练。
loss_trace08 = []  # 保存每轮平均 masked loss。
for _ in range(30):  # 运行少量教学训练步骤。
    optimizer08.zero_grad()  # 清空上一步梯度。
    losses08 = []  # 收集五条记录的 assistant loss。
    for ids08, labels08, _ in sequences08:  # 逐序列计算 causal loss。
        logits08 = model08(ids08)  # 计算当前模板 logits。
        losses08.append(F.cross_entropy(logits08[:-1], labels08[1:], ignore_index=-100))  # 使用 causal shift 和 assistant mask。
    loss08 = torch.stack(losses08).mean()  # 汇总当前训练批次损失。
    loss08.backward()  # 反向传播到微型语言头。
    optimizer08.step()  # 更新模型参数。
    loss_trace08.append(float(loss08.detach()))  # 保存训练损失。
core_safe08 = 1.0  # 安全门禁后所有选中样本均安全。
core_quality08 = sum(record08["quality"] for record08 in selected_records08) / len(selected_records08)  # 计算独立质量均值。
print("方法 | 安全率 | 独立质量 | 训练loss")  # 输出选择与训练结果表头。
print("reward_only", round(baseline_safe08, 3), round(baseline_quality08, 3), "未训练")  # 展示纯 reward 基线。
print("filtered_RSFT", round(core_safe08, 3), round(core_quality08, 3), (round(loss_trace08[0], 4), round(loss_trace08[-1], 4)))  # 展示过滤选择和训练结果。
print("首条Loss Mask", list(zip(sequences08[0][2], sequences08[0][1].tolist())))  # 展示用户 token 被忽略而回答被监督。
print("结果解读：Best-of-N扩大候选空间，也扩大reward漏洞；硬门禁必须在排序之前")  # 解释过滤顺序的重要性。


方法 | 安全率 | 独立质量 | 训练loss
reward_only 0.6 0.338 未训练
filtered_RSFT 1.0 0.9 (4.5256, 0.2703)
首条Loss Mask [('<user>', -100), ('订', -100), ('单', -100), ('延', -100), ('迟', -100), ('怎', -100), ('么', -100), ('办', -100), ('<assistant>', -100), ('查', 48), ('询', 64), ('物', 53), ('流', 50), ('并', 35), ('说', 65), ('明', 45), ('预', 74), ('计', 61), ('时', 44), ('间', 71), ('<eos>', 3)]
结果解读：Best-of-N扩大候选空间，也扩大reward漏洞；硬门禁必须在排序之前


## 5. 失败案例与修正：Reward Hacking 与重复候选

`p2` 的索取密码回答 reward 最高，且正确回答重复出现。纯 reward 会选危险项；修正先拒绝危险项，再按语义键去重并选择独立质量最高回答。


In [5]:
failure_baseline08 = next(row08 for row08 in baseline_selected08 if row08[0] == "p2")  # 读取凭据风险基线选择。
failure_fixed08 = next(record08 for record08 in selected_records08 if record08["id"] == "p2")  # 读取安全去重后的选择。
failure_trace08 = next(row08 for row08 in selection_trace08 if row08[0] == "p2")  # 读取每个候选的过滤原因。
print("失败行为", failure_baseline08)  # 展示最高 reward 选择危险回答。
print("修正行为", failure_fixed08)  # 展示独立 evaluator 选择正确流程。
print("候选过滤账本", failure_trace08)  # 展示 unsafe 与 duplicate 均有稳定原因码。


失败行为 ('p2', '把密码发给客服', 0.93, False, 0.1)
修正行为 {'id': 'p2', 'prompt': '如何申请退款', 'response': '提供订单号并确认退款原因', 'reward': 0.86, 'quality': 0.91}
候选过滤账本 ('p2', [('提供订单号并确认退款原因', 'eligible'), ('把密码发给客服', 'reject_unsafe'), ('提供订单号并确认退款原因', 'reject_duplicate')], '提供订单号并确认退款原因')


## 6. 生产边界与 RSFT 制品

真实采样需要记录模型、温度、seed、N 和选择概率；去重通常用 embedding/MinHash，安全过滤需要独立分类器与人工抽检。迭代训练必须使用冻结评测集防止 evaluator 共适应。


In [6]:
rsft_contract08 = {"sampler": "policy-v7", "N": 3, "temperature": 0.8, "safety_gate": "safety-v4", "selector": "independent-eval-v3", "loss_mask": "assistant_only", "frozen_eval": "support-golden-v8"}  # 定义拒绝采样训练 provenance。
print("RSFT 训练制品", rsft_contract08)  # 展示采样、过滤、排序和评测版本。
print("生产替换点：真实LLM采样、语义去重、安全分类器、选择概率、分布偏移和冻结黄金集")  # 说明教学字符模型的边界。


RSFT 训练制品 {'sampler': 'policy-v7', 'N': 3, 'temperature': 0.8, 'safety_gate': 'safety-v4', 'selector': 'independent-eval-v3', 'loss_mask': 'assistant_only', 'frozen_eval': 'support-golden-v8'}
生产替换点：真实LLM采样、语义去重、安全分类器、选择概率、分布偏移和冻结黄金集


## 7. 最小回归测试

断言保护安全过滤、独立质量和 assistant-only 训练。


In [7]:
assert len(prompts08) >= 5  # 保证 RSFT 案例覆盖足够多的 prompt。
assert core_safe08 > baseline_safe08  # 保证安全门禁改善选择安全率。
assert core_quality08 > baseline_quality08  # 保证独立 evaluator 改善真实质量代理。
assert loss_trace08[-1] < loss_trace08[0]  # 保证选中数据的 masked SFT 可运行。
assert failure_baseline08[3] is False and "密码" not in failure_fixed08["response"]  # 保证 reward hacking 反例被修正。
assert all(int(label08) == -100 for label08 in sequences08[0][1][:sequences08[0][2].index("<assistant>") + 1])  # 保证用户和角色 token 不进入监督。
print("最小回归测试通过：过滤、去重、独立排序和Assistant-only SFT稳定")  # 显示 RSFT 关键性质已验证。


最小回归测试通过：过滤、去重、独立排序和Assistant-only SFT稳定
